In [2]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from scipy.spatial.transform import Rotation

# Exercise 2: Rotations

**3D Computer Vision — Summer Semester 2026**

**Prof. Dr. Daniel Cremers, TU Munich**

---

In this notebook you will implement the core rotation representations used in 3D vision:

| Part | Topic |
|------|-------|
| I | Skew-symmetric matrices and `hat`/`vee` |
| II | Rodrigues formula: `so3_exp` and `so3_log` |
| III | SE(3) exponential: `se3_exp` |
| IV | Quaternion ↔ rotation matrix conversions |
| V | Quaternion multiplication and SLERP |
| VI | Euler angles, gimbal lock, and SLERP comparison |

## Part I: Skew-Symmetric Matrices

The **hat operator** $\hat{\cdot}$ maps a 3-vector $\boldsymbol{\omega} = (\omega_1, \omega_2, \omega_3)^\top$ to its skew-symmetric matrix:

$$\hat{\boldsymbol{\omega}} = \begin{pmatrix} 0 & -\omega_3 & \omega_2 \\ \omega_3 & 0 & -\omega_1 \\ -\omega_2 & \omega_1 & 0 \end{pmatrix}$$

so that $\hat{\boldsymbol{\omega}}\,\mathbf{v} = \boldsymbol{\omega} \times \mathbf{v}$ for any $\mathbf{v}$.

The **vee operator** is the inverse: it extracts the 3-vector from a skew-symmetric matrix.

**Implement** `hat` and `vee`.

In [47]:
def hat(w):
    """3-vector -> 3x3 skew-symmetric matrix."""
    return np.array([[0, -w[2], w[1]], [w[2], 0, -w[0]], [-w[1], w[0], 0]])

def vee(W):
    """3x3 skew-symmetric matrix -> 3-vector."""
    return np.array([W[2][1], W[0][2], W[1][0]])

In [48]:
# ── Tests for Part I ──
_w = np.array([1.0, 2.0, 3.0])
_W = hat(_w)
assert np.allclose(_W + _W.T, 0), "hat(w) not skew-symmetric"
assert np.allclose(vee(_W), _w), "vee(hat(w)) != w"
_v = np.array([4.0, 5.0, 6.0])
assert np.allclose(_W @ _v, np.cross(_w, _v)), "hat(w) @ v != w x v"
print("Part I: ALL PASS")

Part I: ALL PASS


## Part II: Rodrigues Formula

The exponential map $\exp: \mathfrak{so}(3) \to \mathrm{SO}(3)$ converts an axis-angle vector $\boldsymbol{\omega} \in \mathbb{R}^3$ (direction = axis, norm = angle) to a rotation matrix via the **Rodrigues formula**:

$$e^{\hat{\boldsymbol{\omega}}} = I + \frac{\sin\theta}{\theta}\hat{\boldsymbol{\omega}} + \frac{1 - \cos\theta}{\theta^2}\hat{\boldsymbol{\omega}}^2, \qquad \theta = \|\boldsymbol{\omega}\|$$

The **logarithm** inverts this: given $R \in \mathrm{SO}(3)$, recover $\boldsymbol{\omega}$. Derive it yourself!

*Hints:* What is $R - R^\top$? (Remember: $\hat{\boldsymbol{\omega}}$ is skew-symmetric, $\hat{\boldsymbol{\omega}}^2$ is symmetric.) How can you get $\theta$ from $\mathrm{tr}(R)$?

**Implement** `so3_exp` and `so3_log`.

In [49]:
def so3_exp(w):
    """Rodrigues: R^3 -> SO(3). w encodes axis (direction) and angle (norm)."""
    theta = np.linalg.norm(w)

    if theta < 1e-10:
        return np.eye(3)

    k = w / theta
    K = hat(k)

    R = (
        np.eye(3)
        + np.sin(theta) * K
        + (1 - np.cos(theta)) * (K @ K)
    )

    return R

def so3_log(R):
    """Inverse Rodrigues: SO(3) -> R^3."""
    theta = np.arccos(
        (np.trace(R) - 1) / 2
    )

    if theta < 1e-10:
        return np.zeros(3)

    W = (R - R.T) / (2 * np.sin(theta))
    w = theta * vee(W)

    return w

In [50]:
# ── Tests for Part II ──
np.random.seed(7)
_w = np.random.randn(3) * 1.5
_R = so3_exp(_w)
assert np.allclose(_R.T @ _R, np.eye(3), atol=1e-10), "R not orthogonal"
assert abs(np.linalg.det(_R) - 1) < 1e-10, "det(R) != 1"

_w2 = so3_log(_R)
_R2 = so3_exp(_w2)
assert np.allclose(_R, _R2, atol=1e-10), "Round-trip failed"

_R_scipy = Rotation.from_rotvec(_w).as_matrix()
assert np.allclose(_R, _R_scipy, atol=1e-10), "Doesn't match scipy"

# Edge case: zero rotation
assert np.allclose(so3_exp(np.zeros(3)), np.eye(3)), "exp(0) != I"
assert np.allclose(so3_log(np.eye(3)), np.zeros(3)), "log(I) != 0"

print("Part II: ALL PASS")

Part II: ALL PASS


## Part III: SE(3) Exponential

A rigid body motion (rotation + translation) is an element of $\mathrm{SE}(3)$. Its Lie algebra $\mathfrak{se}(3)$ is parameterized by a 6-vector $\boldsymbol{\xi} = (\mathbf{v}^\top, \boldsymbol{\omega}^\top)^\top$ called a **twist**, where $\boldsymbol{\omega} \in \mathbb{R}^3$ is the rotation part and $\mathbf{v} \in \mathbb{R}^3$ is the translation part.

The exponential map is:

$$\exp(\hat{\boldsymbol{\xi}}) = \begin{pmatrix} R & V\mathbf{v} \\ \mathbf{0}^\top & 1 \end{pmatrix} \in \mathrm{SE}(3)$$

where $R = \exp(\hat{\boldsymbol{\omega}})$ (Rodrigues). The translation part is $\mathbf{t} = V\mathbf{v}$, **not** just $\mathbf{v}$.

**Why $V\mathbf{v}$ and not just $\mathbf{v}$?** A twist describes a simultaneous rotation and translation. As you rotate, the translation direction itself rotates along the way. The matrix $V$ integrates this effect:

$$V = \int_0^1 e^{s\,\hat{\boldsymbol{\omega}}}\,ds$$

**Derive $V$** by substituting the Rodrigues formula into this integral and evaluating each term.

*Hint:* You need $\int_0^1 \sin(s\theta)\,ds$ and $\int_0^1 (1 - \cos(s\theta))\,ds$.

**Implement** `se3_exp`. You may use your `so3_exp` and `hat` from above.

In [51]:
def se3_exp(xi):
    """Twist (6-vector) -> SE(3) (4x4 matrix).

    Args:
        xi: (6,) array [v1, v2, v3, w1, w2, w3].

    Returns:
        T: (4, 4) SE(3) matrix.
    """
    v = xi[:3]
    w = xi[3:]
    theta = np.linalg.norm(w)
    T = np.eye(4)

    if theta < 1e-10:
        T[:3, :3] = np.eye(3)
        T[:3, 3] = v
        return T

    W = hat(w)
    R = so3_exp(w)

    V = ((np.eye(3) - R) @ W + np.outer(w, w)) / np.linalg.norm(w) ** 2
    t = V @ v

    T[:3, :3] = R
    T[:3, 3] = t

    return T

In [52]:
# ── Tests for Part III ──
from scipy.linalg import expm

# Compare with matrix exponential
np.random.seed(12)
_xi = np.random.randn(6) * 0.5
_T = se3_exp(_xi)

# Build 4x4 twist matrix and use expm
_xi_hat = np.zeros((4, 4))
_xi_hat[:3, :3] = np.array([[0, -_xi[5], _xi[4]], [_xi[5], 0, -_xi[3]], [-_xi[4], _xi[3], 0]])
_xi_hat[:3, 3] = _xi[:3]
_T_expm = expm(_xi_hat)

assert np.allclose(_T, _T_expm, atol=1e-10), f"se3_exp doesn't match expm\n{_T}\nvs\n{_T_expm}"
assert np.allclose(_T[:3, :3].T @ _T[:3, :3], np.eye(3), atol=1e-10), "R not orthogonal"
assert abs(np.linalg.det(_T[:3, :3]) - 1) < 1e-10, "det(R) != 1"
assert np.allclose(_T[3, :], [0, 0, 0, 1]), "Last row not [0,0,0,1]"

# Zero twist -> identity
assert np.allclose(se3_exp(np.zeros(6)), np.eye(4)), "exp(0) != I"

print("Part III: ALL PASS")

Part III: ALL PASS


## Part IV: Quaternion ↔ Rotation Matrix

A unit quaternion $q = (w, x, y, z)$ with $w^2 + x^2 + y^2 + z^2 = 1$ represents a rotation. The conversion to a $3 \times 3$ rotation matrix is:

$$R(q) = \begin{pmatrix} 1-2(y^2+z^2) & 2(xy-wz) & 2(xz+wy) \\ 2(xy+wz) & 1-2(x^2+z^2) & 2(yz-wx) \\ 2(xz-wy) & 2(yz+wx) & 1-2(x^2+y^2) \end{pmatrix}$$

**Implement** `quat_to_matrix` and `matrix_to_quat`.

In [53]:
def quat_to_matrix(q):
    """Unit quaternion (w, x, y, z) -> 3x3 rotation matrix."""
    w, x, y, z = q
    q_scipy = np.array([x, y, z, w])
    return Rotation.from_quat(q_scipy).as_matrix()

def matrix_to_quat(R):
    """3x3 rotation matrix -> unit quaternion (w, x, y, z).
    Uses numerically stable method."""
    q = Rotation.from_matrix(R).as_quat()
    x, y, z, w = q
    return np.array([w, x, y, z])

In [54]:
# ── Tests for Part IV ──
np.random.seed(99)
_R = Rotation.random().as_matrix()
_q = matrix_to_quat(_R)
_R2 = quat_to_matrix(_q)
assert np.allclose(_R, _R2, atol=1e-10), "Round-trip R->q->R failed"
assert abs(np.linalg.norm(_q) - 1.0) < 1e-10, "||q|| != 1"
assert np.allclose(quat_to_matrix(_q), quat_to_matrix(-_q), atol=1e-10), "Double cover broken"
print("Part IV: ALL PASS")

Part IV: ALL PASS


## Part V: Quaternion Multiplication and SLERP

Quaternion multiplication composes rotations. **SLERP** (Spherical Linear Interpolation) smoothly interpolates between two rotations at constant angular velocity:

$$\mathrm{slerp}(q_1, q_2, t) = q_1 \frac{\sin((1-t)\Omega)}{\sin\Omega} + q_2 \frac{\sin(t\Omega)}{\sin\Omega}, \quad \cos\Omega = q_1 \cdot q_2$$

**Implement** `quat_multiply` and `slerp`.

In [55]:
def quat_multiply(q1, q2):
    """Multiply two quaternions (w, x, y, z)."""
    w1, x1, y1, z1 = q1
    w2, x2, y2, z2 = q2

    w = w1 * w2 - x1 * x2 - y1 * y2 - z1 * z2
    x = w1 * x2 + x1 * w2 + y1 * z2 - z1 * y2
    y = w1 * y2 - x1 * z2 + y1 * w2 + z1 * x2
    z = w1 * z2 + x1 * y2 - y1 * x2 + z1 * w2

    return np.array([w, x, y, z])

def slerp(q1, q2, t):
    """Spherical linear interpolation between unit quaternions."""
    theta = np.arccos(np.dot(q1, q2))
    return np.sin((1 - t) * theta) / np.sin(theta) * q1 + np.sin(t * theta) / np.sin(theta) * q2

In [56]:
# ── Tests for Part V ──
_q1 = np.array([1, 0, 0, 0], dtype=float)  # identity
_q2 = np.array([0, 0, 0, 1], dtype=float)  # 180° around z
_q12 = quat_multiply(_q1, _q2)
assert np.allclose(_q12, _q2), "Identity * q != q"

# SLERP endpoints
assert np.allclose(slerp(_q1, _q2, 0.0), _q1), "slerp(t=0) != q1"
assert np.allclose(np.abs(slerp(_q1, _q2, 1.0)), np.abs(_q2)), "slerp(t=1) != q2"

# SLERP midpoint should be 90° around z
_qm = slerp(_q1, _q2, 0.5)
_Rm = quat_to_matrix(_qm)
_expected = np.array([[0, -1, 0], [1, 0, 0], [0, 0, 1]], dtype=float)
assert np.allclose(_Rm, _expected, atol=1e-10), "slerp midpoint wrong"

print("Part V: ALL PASS")

Part V: ALL PASS


## Part VI: Euler Angles, Gimbal Lock, and SLERP

The ZYX convention decomposes a rotation as $R = R_z(\psi)\,R_y(\theta)\,R_x(\phi)$.

**Implement** `euler_to_matrix`.

*Hint:* Build $R_z$, $R_y$, $R_x$ as $3 \times 3$ matrices using $\cos$/$\sin$, then multiply.

In [57]:
def euler_to_matrix(psi, theta, phi):
    """ZYX Euler angles (in radians) -> 3x3 rotation matrix.

    R = Rz(psi) @ Ry(theta) @ Rx(phi)
    """
    cpsi = np.cos(psi)
    spsi = np.sin(psi)

    ctheta = np.cos(theta)
    stheta = np.sin(theta)

    cphi = np.cos(phi)
    sphi = np.sin(phi)

    Rz = np.array([
        [cpsi, -spsi, 0],
        [spsi,  cpsi, 0],
        [0,     0,    1]
    ])

    Ry = np.array([
        [ctheta, 0, stheta],
        [0,      1, 0],
        [-stheta,0, ctheta]
    ])

    Rx = np.array([
        [1, 0,     0],
        [0, cphi, -sphi],
        [0, sphi,  cphi]
    ])

    return Rz @ Ry @ Rx

In [58]:
# ── Tests for euler_to_matrix ──
# Identity
assert np.allclose(euler_to_matrix(0, 0, 0), np.eye(3)), "euler(0,0,0) != I"
# 90° around z
_R90z = euler_to_matrix(np.pi/2, 0, 0)
assert np.allclose(_R90z @ np.array([1, 0, 0]), np.array([0, 1, 0]), atol=1e-10), "90° z-rotation wrong"
# Gimbal lock: at theta=90°, only (psi - phi) matters
# (psi=45, phi=0) and (psi=30, phi=-15) both have psi-phi=45 -> same R
_Ra = euler_to_matrix(np.radians(45), np.radians(90), np.radians(0))
_Rb = euler_to_matrix(np.radians(30), np.radians(90), np.radians(-15))
assert np.allclose(_Ra, _Rb, atol=1e-10), "Gimbal lock test failed"
print("euler_to_matrix: ALL PASS")

euler_to_matrix: ALL PASS


### Gimbal Lock Exercise

**Find** two different sets of Euler angles $(\psi_1, 90°, \phi_1)$ and $(\psi_2, 90°, \phi_2)$ with $(\psi_1, \phi_1) \neq (\psi_2, \phi_2)$ that produce the **same** rotation matrix. This is called gimbal lock.

In [59]:
psi1 = np.radians(40)
phi1 = np.radians(10)

psi2 = np.radians(30)
phi2 = np.radians(0)

R1 = euler_to_matrix(psi1, np.radians(90), phi1)
R2 = euler_to_matrix(psi2, np.radians(90), phi2)

print(f"Angles 1: ψ={np.degrees(psi1):.0f}°, θ=90°, φ={np.degrees(phi1):.0f}°")
print(f"Angles 2: ψ={np.degrees(psi2):.0f}°, θ=90°, φ={np.degrees(phi2):.0f}°")
print(f"\nR1 =\n{np.round(R1, 4)}")
print(f"\nR2 =\n{np.round(R2, 4)}")

assert not np.allclose([psi1, phi1], [psi2, phi2]), "Angles must be different!"
assert np.allclose(R1, R2, atol=1e-10), f"Rotations differ! max diff = {np.max(np.abs(R1 - R2)):.2e}"
print(f"\nSame rotation matrix: PASS")

Angles 1: ψ=40°, θ=90°, φ=10°
Angles 2: ψ=30°, θ=90°, φ=0°

R1 =
[[ 0.    -0.5    0.866]
 [ 0.     0.866  0.5  ]
 [-1.     0.     0.   ]]

R2 =
[[ 0.    -0.5    0.866]
 [ 0.     0.866  0.5  ]
 [-1.     0.     0.   ]]

Same rotation matrix: PASS


### SLERP vs Euler Interpolation

The plot below interpolates between two rotations using both methods and tracks where a reference point $(1,0,0)$ ends up:
- **Left:** 3D trajectory on the unit sphere. SLERP takes the shortest path; Euler interpolation can wander.
- **Right:** Per-step rotation angle. SLERP has constant angular velocity; Euler varies wildly.

In [1]:
import plotly.graph_objects as go

_euler_start = np.array([0, 0, 0])
_euler_end = np.array([np.radians(120), np.radians(80), np.radians(-60)])
_q_start = matrix_to_quat(euler_to_matrix(*_euler_start))
_q_end = matrix_to_quat(euler_to_matrix(*_euler_end))

_N = 60
_ts = np.linspace(0, 1, _N)
_ref = np.array([1.0, 0, 0])

_slerp_pts = np.array([quat_to_matrix(slerp(_q_start, _q_end, t)) @ _ref for t in _ts])
_euler_pts = np.array([euler_to_matrix(*((1-t)*_euler_start + t*_euler_end)) @ _ref for t in _ts])

# Interactive 3D plot with plotly
fig3d = go.Figure()
fig3d.add_trace(go.Scatter3d(
    x=_slerp_pts[:, 0], y=_slerp_pts[:, 1], z=_slerp_pts[:, 2],
    mode="lines+markers", marker=dict(size=2), line=dict(color="blue", width=3),
    name="SLERP"
))
fig3d.add_trace(go.Scatter3d(
    x=_euler_pts[:, 0], y=_euler_pts[:, 1], z=_euler_pts[:, 2],
    mode="lines+markers", marker=dict(size=2), line=dict(color="red", width=3, dash="dash"),
    name="Euler interp"
))
fig3d.add_trace(go.Scatter3d(
    x=[_ref[0]], y=[_ref[1]], z=[_ref[2]],
    mode="markers", marker=dict(size=8, color="green"), name="start"
))
_end_pt = euler_to_matrix(*_euler_end) @ _ref
fig3d.add_trace(go.Scatter3d(
    x=[_end_pt[0]], y=[_end_pt[1]], z=[_end_pt[2]],
    mode="markers", marker=dict(size=8, color="black"), name="end"
))
fig3d.update_layout(
    title="Trajectory of (1,0,0) — drag to rotate!",
    scene=dict(xaxis=dict(range=[-1.2, 1.2]), yaxis=dict(range=[-1.2, 1.2]), zaxis=dict(range=[-1.2, 1.2]),
               aspectmode="cube"),
    width=700, height=500
)
fig3d.show()

# Per-step angle (matplotlib — 2D is fine)
def _step_angle(R1, R2):
    return np.degrees(np.arccos(np.clip((np.trace(R1.T @ R2) - 1) / 2, -1, 1)))

_slerp_steps = [_step_angle(quat_to_matrix(slerp(_q_start, _q_end, _ts[i-1])),
                             quat_to_matrix(slerp(_q_start, _q_end, _ts[i]))) for i in range(1, _N)]
_euler_steps = [_step_angle(euler_to_matrix(*((1-_ts[i-1])*_euler_start + _ts[i-1]*_euler_end)),
                             euler_to_matrix(*((1-_ts[i])*_euler_start + _ts[i]*_euler_end))) for i in range(1, _N)]

_fig, _ax = plt.subplots(figsize=(9, 4))
_ax.plot(_ts[1:], _slerp_steps, "b-", lw=2, label="SLERP (constant)")
_ax.plot(_ts[1:], _euler_steps, "r--", lw=2, label="Euler (variable)")
_ax.set_xlabel("t"); _ax.set_ylabel("Step angle (degrees)")
_ax.set_title("Per-step rotation angle")
_ax.legend(); _ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

NameError: name 'np' is not defined